## Import libraries

In [13]:
import arcpy
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os

### Read a construction points CSV file, filter by municipality == Sofia, extract only latitude and longitude columns 

In [23]:
# Relative, not absolute path
read_file_construction_points = pd.read_csv(".\\construction_site_points_utm35n.csv", sep=",")

# Filter by city = Sofia, get a projection only of the columns that you actually need (municipality becomes useless after the filter)
read_file_sofia_based_construction_points = read_file_construction_points[read_file_construction_points["municipality"].str.lower() == "софия"][["latitude", "longitude"]]

read_file_sofia_based_construction_points.to_csv(".\\sofia_based_construction_site_points.csv", index=False)

read_file_sofia_based_construction_points.head(5)

,latitude,longitude
0,4733808.208,198721.721
1,4732568.941,198306.502
2,4736258.328,200206.923
3,4728772.418,202469.297
4,4737948.299,196925.371


### Create a point layer, based on the previous step result CSV file

In [22]:
UTM_35N_CODE = 32635

# Work with relative path, not absolute
current_working_directory_path = os.getcwd()

# Create a point layer from the construction coordiantes in Sofia
arcpy.management.XYTableToPoint(
    in_table=os.path.join(current_working_directory_path, "sofia_based_construction_site_points.csv"),
    out_feature_class=os.path.join(current_working_directory_path, "HandsOnTrainingWithPython.gdb", "Sofia_based_construction_points"),
    x_field="longitude",
    y_field="latitude",
    z_field=None,
    coordinate_system=arcpy.SpatialReference(UTM_35N_CODE)
)

<Result 'C:\\Users\\HP ZBook 17 G5\\Documents\\ArcGIS\\Projects\\HandsOnTrainingWithPython\\HandsOnTrainingWithPython.gdb\\Sofia_based_construction_points'>

### Calculate new land_area_name column, based on the point's position

In [24]:
sofia_based_construction_points_layer = os.path.join(
    os.getcwd(), 
    "HandsOnTrainingWithPython.gdb", 
    "Sofia_based_construction_points"
)

arcpy.management.AddField(
    sofia_based_construction_points_layer, 
    "land_area_name", 
    "TEXT", 
    field_alias="Име на землището", 
    field_length=100
)

sofia_land_areas_layer = os.path.join(
    os.getcwd(),
    "HandsOnTrainingWithPython.gdb",
    "Sofia_Land_Areas"
)

# Create a temporary layer of the JOIN operation
arcpy.analysis.SpatialJoin(
    target_features=sofia_based_construction_points_layer,
    join_features=sofia_land_areas_layer,
    out_feature_class="temp_join",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT" # We don't use WITHIN (must be fully inside the polygon. Points that fall exactly on a polygon boundary are excluded) [we have such case]
)

lookup = {}
with arcpy.da.SearchCursor("temp_join", ["TARGET_FID", "Name_bg"]) as cursor:
    for row in cursor:
        lookup[row[0]] = row[1]

with arcpy.da.UpdateCursor(sofia_based_construction_points_layer, ["OID@", "land_area_name"]) as cursor:
    for row in cursor:
        row[1] = lookup.get(row[0])
        cursor.updateRow(row)

# Delete the temporary layer, we don't need it anymore
arcpy.management.Delete("temp_join")

<Result 'true'>

### Calculate new distance_to_road_meters column, based on how far a point is from the nearest road

In [30]:
arcpy.management.AddField(
    sofia_based_construction_points_layer,
    "distance_to_road_meters",
    "Double",
    field_alias="Дистанция до път (метри)",
    field_scale=3
)

sofia_roads_layer = os.path.join(
    os.getcwd(),
    "HandsOnTrainingWithPython.gdb",
    "Sofia_Roads"
)

arcpy.analysis.Near(
    in_features=sofia_based_construction_points_layer,
    near_features=sofia_roads_layer,
    search_radius=None,
    location="NO_LOCATION",
    angle="NO_ANGLE",
    method="PLANAR"
)

with arcpy.da.UpdateCursor(sofia_based_construction_points_layer, ["NEAR_DIST", "distance_to_road_meters"]) as cursor:
    for row in cursor:
        row[1] = row[0]
        cursor.updateRow(row)

arcpy.management.DeleteField(sofia_based_construction_points_layer, ["NEAR_DIST", "NEAR_FID"])

1007


<Result 'C:\\Users\\HP ZBook 17 G5\\Documents\\ArcGIS\\Projects\\HandsOnTrainingWithPython\\HandsOnTrainingWithPython.gdb\\Sofia_based_construction_points'>